# 01 - Dataset Audit

**Purpose.** Verify that the local IDC dataset matches the binary task used by the code: class `0` for non-IDC patches and class `1` for IDC-positive patches.

**Research integrity rule.** This notebook explains the data. The reproducible source of truth is the saved split file and the experiment config used by `dcpgann-train`.

In [ ]:
from collections import Counter
from pathlib import Path
import json

from PIL import Image
from IPython.display import display
from torchvision import datasets

from dcpgann.data import DataConfig, IDCDataModule

CONFIG_PATH = Path('../configs/paper_2022_idc.json')
config = json.loads(CONFIG_PATH.read_text())

# Override this if your data lives elsewhere.
DATA_DIR = Path(config['data']['data_dir'])
if not DATA_DIR.exists():
    DATA_DIR = Path('../data/idc')

print('Dataset path:', DATA_DIR.resolve())
print('Exists:', DATA_DIR.exists())

## Folder Contract

The loader expects this layout:

```text
data/idc/
  0/
  1/
```

In [ ]:
if not DATA_DIR.exists():
    print('Dataset not found. Download and prepare it first; see README.md.')
else:
    dataset = datasets.ImageFolder(DATA_DIR)
    counts = Counter(dataset.targets)
    idx_to_class = {idx: name for name, idx in dataset.class_to_idx.items()}
    print('class_to_idx:', dataset.class_to_idx)
    print('total images:', len(dataset))
    for idx, count in sorted(counts.items()):
        print(f'class {idx_to_class[idx]}: {count:,} images ({count / len(dataset):.2%})')

## Visual Sanity Check

A few sample patches should render as small histopathology tiles. This is not a model result; it is a guard against path/layout mistakes.

In [ ]:
if DATA_DIR.exists():
    for label in ['0', '1']:
        print(f'Class {label}')
        examples = sorted((DATA_DIR / label).glob('*.png'))[:6]
        thumbs = []
        for path in examples:
            image = Image.open(path).convert('RGB').resize((90, 90))
            thumbs.append(image)
        if thumbs:
            canvas = Image.new('RGB', (90 * len(thumbs), 90), 'white')
            for i, image in enumerate(thumbs):
                canvas.paste(image, (90 * i, 0))
            display(canvas)
        else:
            print('No PNG examples found.')

## Reproducible Split Audit

The following cell creates or loads the split file using the same `IDCDataModule` used by the CLI. Preserve the generated `splits.json` when reporting results.

In [ ]:
if DATA_DIR.exists():
    data_cfg = DataConfig(
        data_dir=DATA_DIR,
        batch_size=config['data'].get('batch_size', 64),
        num_workers=0,
        val_split=config['data'].get('val_split', 0.15),
        test_split=config['data'].get('test_split', 0.15),
        image_size=config['data'].get('image_size', 224),
        seed=config.get('seed', 42),
        split_file=Path('../artifacts/notebook_dataset_audit/splits.json'),
        patient_level_split=config['data'].get('patient_level_split', False),
    )
    loaders = IDCDataModule(data_cfg).setup()
    print('split sizes:', {
        'train': len(loaders.splits.train),
        'val': len(loaders.splits.val),
        'test': len(loaders.splits.test),
    })
    print('saved split file:', data_cfg.split_file.resolve())